# arms_embed

Deterministic identity measures on the adapted arms: DINOv2-large, CLIP-large and ArcFace
similarity between each reference and the generated image, on the 488-row test split.

The baseline was measured by `baseline_embed` and `baseline_arcface` and is not recomputed;
those two files are the comparator. This notebook adds four arms: LoRA r8 at step 1000 and at
step 4000, QLoRA r8 at step 2000, and soft prompt (iii) at step 2000.

Purpose is an instrument independent of GPT-4.1, not a proxy for it. The earlier finding that
these measures do not track GPT SC row by row (DINO r 0.04, CLIP r 0.11, ArcFace 0.23) is a
rank-correlation result and does not bear on whether they detect the same arm-level mean shift.

ArcFace coverage is uneven with respect to where the LoRA damage sits: 54% broad on `all_up`
rows, which carry SC −0.540 at step 1000, against 87% on `no_up` and 100% on `mixed`. DINOv2 and
CLIP cover 488 of 488. The embedding measures are therefore the primary instrument here and
ArcFace is a secondary check on the face-bearing rows.

No API calls. Reference embeddings are computed once and shared across arms: 358 unique
references against 784 instances, so the per-arm cost is 488 generated images.

T4 is sufficient. DINOv2 and CLIP are the only GPU work; insightface ran on CPU in
`baseline_arcface` because onnxruntime did not expose a CUDA provider, and section 1 reports
which provider it gets.

## 1. Setup

In [1]:
from google.colab import drive
import os, sys, json, subprocess

drive.mount("/content/drive")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "insightface", "onnxruntime-gpu"], check=False)

DRIVE  = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
SCORES = f"{DRIVE}/scores"
IMGS, EXPECTED = "/content/imgs", 37063

def n_files(d):
    return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    import shutil
    free_gb = shutil.disk_usage("/content").free / 1e9
    print(f"free before restore: {free_gb:.0f} GB")
    assert free_gb > 55, f"only {free_gb:.0f} GB free; the extracted image set needs ~50 GB"
    os.makedirs(IMGS, exist_ok=True)
    # extracted straight from Drive: copying the tar locally first would hold 48 GB of archive
    # and 48 GB of extracted files at the same time, which overflows the Colab disk
    subprocess.run(f'tar -xf "{DRIVE}/imgs_subset.tar" -C {IMGS}', shell=True, check=True)
    assert n_files(IMGS) >= EXPECTED, f"restore incomplete: {n_files(IMGS)} of {EXPECTED}"
print("reference images:", n_files(IMGS))

if not os.path.exists("/content/test.jsonl"):
    subprocess.run(["cp", f"{DRIVE}/test.jsonl", "/content/test.jsonl"], check=True)
rows = [json.loads(l) for l in open("/content/test.jsonl", encoding="utf-8")]
assert len(rows) == 488, f"expected 488 test rows, got {len(rows)}"

import torch, torch.nn.functional as F, numpy as np
from PIL import Image
from transformers import AutoImageProcessor, AutoModel, CLIPImageProcessor, CLIPVisionModelWithProjection

device = "cuda" if torch.cuda.is_available() else "cpu"
dino_proc = AutoImageProcessor.from_pretrained("facebook/dinov2-large")
dino      = AutoModel.from_pretrained("facebook/dinov2-large").to(device).eval()
clip_proc = CLIPImageProcessor.from_pretrained("openai/clip-vit-large-patch14")
clip      = CLIPVisionModelWithProjection.from_pretrained("openai/clip-vit-large-patch14").to(device).eval()

@torch.no_grad()
def dino_emb(imgs):                       # list[PIL] -> (N,D) L2-normalized CLS tokens
    inp = dino_proc(images=imgs, return_tensors="pt").to(device)
    return F.normalize(dino(**inp).pooler_output.float(), dim=-1)

@torch.no_grad()
def clip_emb(imgs):                       # list[PIL] -> (N,D) L2-normalized projected image features
    inp = clip_proc(images=imgs, return_tensors="pt").to(device)
    return F.normalize(clip(**inp).image_embeds.float(), dim=-1)

# insightface's own downloader does not retry and resolves a different release tag depending on
# its version; baseline_arcface ran on v0.7, so that build is fetched explicitly and the model
# is identical to the one behind baseline_arcface.jsonl
IF_DIR = os.path.expanduser("~/.insightface/models")
if not os.path.isdir(f"{IF_DIR}/buffalo_l"):
    os.makedirs(IF_DIR, exist_ok=True)
    subprocess.run(["rm", "-rf", f"{IF_DIR}/buffalo_l", f"{IF_DIR}/buffalo_l.zip"], check=False)
    subprocess.run(["curl", "-L", "--retry", "5", "--retry-all-errors", "--fail",
                    "-o", f"{IF_DIR}/buffalo_l.zip",
                    "https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip"],
                   check=True)
    subprocess.run(["unzip", "-q", "-o", f"{IF_DIR}/buffalo_l.zip", "-d", f"{IF_DIR}/buffalo_l"], check=True)
print("buffalo_l:", sorted(f for f in os.listdir(f"{IF_DIR}/buffalo_l") if f.endswith(".onnx")))

from insightface.app import FaceAnalysis
app = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(640, 640))

def to_bgr(pil):
    return np.array(pil.convert("RGB"))[:, :, ::-1]

def largest(faces):
    return max(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1])) if faces else None

print("device:", device,
      "|", torch.cuda.get_device_name(0) if device == "cuda" else "cpu only",
      "| dino dim:", dino.config.hidden_size,
      "| clip dim:", clip.visual_projection.out_features)
print("insightface providers:", app.models["recognition"].session.get_providers())

Mounted at /content/drive
free before restore: 201 GB
reference images: 37063


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/549 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{

buffalo_l: ['1k3d68.onnx', '2d106det.onnx', 'det_10g.onnx', 'genderage.onnx', 'w600k_r50.onnx']


/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


device: cuda | Tesla T4 | dino dim: 1024 | clip dim: 768
insightface providers: ['CPUExecutionProvider']


## 2. Arms

Generated-image directories and their GPT score files. The baseline writes
`test_{i:05d}.png` under `gen/baseline`; the arms write `{i:05d}.png` under `gen/test/<dir>`,
and three of the four directory names carry a double underscore that the score filenames do
not. Both filename patterns are resolved by probing rather than assumed.

In [2]:
ARMS = [
    ("baseline",             "baseline",                 "test__baseline"),
    ("lora_r8_step1000",     "test/lora_r8_step1000",    "test__lora_r8_step1000"),
    ("lora_r8_step4000",     "test/lora_r8",             "test__lora_r8"),
    ("qlora_r8_step2000",    "test/qlora_r8__step2000",  "test__qlora_r8_step2000"),
    ("sp_iii_p139_step2000", "test/sp_iii_p139__step2000", "test__sp_iii_p139_step2000"),
]

def gen_path(gdir, i):
    for pat in (f"{i:05d}.png", f"test_{i:05d}.png"):
        p = f"{DRIVE}/gen/{gdir}/{pat}"
        if os.path.exists(p):
            return p
    return None

print(f"{'arm':24s} {'gen dir':30s} {'pngs':>6s} {'pattern':>16s} {'gpt file':>34s}")
for lab, gdir, sfile in ARMS:
    d = f"{DRIVE}/gen/{gdir}"
    n = len([f for f in os.listdir(d) if f.endswith(".png")]) if os.path.isdir(d) else 0
    p0 = gen_path(gdir, 0)
    pat = os.path.basename(p0).replace("00000", "{i:05d}") if p0 else "MISSING"
    ok = os.path.exists(f"{SCORES}/{sfile}.jsonl")
    print(f"{lab:24s} {gdir:30s} {n:>6d} {pat:>16s} {sfile + ('' if ok else '  MISSING'):>34s}")
    assert n >= 488, f"{lab}: {n} pngs"
    assert p0 is not None, f"{lab}: no image for row 0"
    assert ok, f"{lab}: score file missing"

uid = lambda r: r["output_image"] + "|" + "|".join(sorted(r["input_images"]))
UID = {i: uid(r) for i, r in enumerate(rows)}
REF = {i: [os.path.join(IMGS, p) for p in r["input_images"]] for i, r in enumerate(rows)}
uniq = sorted({p for v in REF.values() for p in v})
print(f"\n{len(rows)} rows | {sum(len(v) for v in REF.values())} reference instances "
      f"| {len(uniq)} unique references")

arm                      gen dir                          pngs          pattern                           gpt file
baseline                 baseline                          488 test_{i:05d}.png                     test__baseline
lora_r8_step1000         test/lora_r8_step1000             488      {i:05d}.png             test__lora_r8_step1000
lora_r8_step4000         test/lora_r8                      488      {i:05d}.png                      test__lora_r8
qlora_r8_step2000        test/qlora_r8__step2000           488      {i:05d}.png            test__qlora_r8_step2000
sp_iii_p139_step2000     test/sp_iii_p139__step2000        488      {i:05d}.png         test__sp_iii_p139_step2000

488 rows | 784 reference instances | 358 unique references


## 3. Reference cache and parity gate

Each unique reference is embedded once and reused across arms. Batching does not couple images,
so a cached embedding is the same tensor the per-row call in `baseline_embed` produced; the gate
below re-derives the first 20 baseline rows with the cached path and compares against
`baseline_embed.jsonl` and `baseline_arcface.jsonl`. A non-trivial difference means the arm
numbers would not be comparable to the baseline files and the run should stop.

In [3]:
import time

t0 = time.time()
if "REF_DINO" not in dir() or len(REF_DINO) != len(uniq):
    REF_DINO, REF_CLIP, REF_FACE = {}, {}, {}
for k, p in enumerate(uniq if len(REF_DINO) != len(uniq) else []):
    im = Image.open(p).convert("RGB")
    REF_DINO[p] = dino_emb([im])[0]
    REF_CLIP[p] = clip_emb([im])[0]
    fl = app.get(to_bgr(im))
    lf = largest(fl)
    REF_FACE[p] = (None if lf is None else lf.normed_embedding, len(fl))
    if (k + 1) % 50 == 0:
        print(f"  {k+1}/{len(uniq)} refs  {time.time()-t0:.0f}s", flush=True)
print(f"reference cache built in {time.time()-t0:.0f}s")

def row_metrics(i, gpath):
    im = Image.open(gpath).convert("RGB")
    gd, gc = dino_emb([im])[0], clip_emb([im])[0]
    refs = REF[i]
    dino_s = float(torch.stack([gd @ REF_DINO[p] for p in refs]).mean())
    clip_s = float(torch.stack([gc @ REF_CLIP[p] for p in refs]).mean())

    gen_faces = app.get(to_bgr(im))
    gen_embs = [f.normed_embedding for f in gen_faces]
    # BROAD: each ref with a face -> its largest face best-matched to any generated face, averaged
    sims = [max(float(REF_FACE[p][0] @ ge) for ge in gen_embs)
            for p in refs if REF_FACE[p][0] is not None and gen_embs]
    broad = sum(sims) / len(sims) if sims else None
    # STRICT: 1 ref, exactly 1 face in ref AND 1 face in generated (unambiguous, no matching)
    strict = None
    if len(refs) == 1 and REF_FACE[refs[0]][1] == 1 and len(gen_faces) == 1:
        strict = float(REF_FACE[refs[0]][0] @ gen_embs[0])
    return dict(dino=dino_s, clip=clip_s, arcface_broad=broad, arcface_strict=strict,
                ref_faces=[REF_FACE[p][1] for p in refs], gen_faces=len(gen_faces))

be = {json.loads(l)["uid"]: json.loads(l) for l in open(f"{SCORES}/baseline_embed.jsonl", encoding="utf-8")}
ba = {json.loads(l)["uid"]: json.loads(l) for l in open(f"{SCORES}/baseline_arcface.jsonl", encoding="utf-8")}
worst = {"dino": 0.0, "clip": 0.0, "arcface_broad": 0.0, "arcface_strict": 0.0}
signed = {k: [] for k in worst}
for i in range(20):
    m = row_metrics(i, gen_path("baseline", i))
    u = UID[i]
    for key, ref in (("dino", be[u]), ("clip", be[u]),
                     ("arcface_broad", ba[u]), ("arcface_strict", ba[u])):
        a, b = m[key], ref[key]
        if a is not None and b is not None:
            worst[key] = max(worst[key], abs(a - b))
            signed[key].append(a - b)
        else:
            assert (a is None) == (b is None), f"row {i} {key}: {a} vs {b}"
print("reproduction of the baseline files, 20 rows, signed mean and max absolute difference:")
for k in worst:
    d = np.array(signed[k], float)
    print(f"  {k:16s} mean {d.mean():+.3e}  max |d| {worst[k]:.3e}  n {len(d)}")
# DINOv2 and CLIP are torch forwards and reproduce to float noise. ArcFace runs through
# onnxruntime, whose build differs with the Python version Colab ships, so it reproduces two
# orders of magnitude looser. The baseline is therefore rescored in this session rather than
# read from the earlier files, and every paired delta below stays within one environment.
assert max(worst["dino"], worst["clip"]) < 1e-3, "embedding path diverges from the baseline files"
print("embedding path reproduces the baseline files; ArcFace is rescored in section 4")

  50/358 refs  77s
  100/358 refs  121s
  150/358 refs  169s
  200/358 refs  215s
  250/358 refs  264s
  300/358 refs  306s
  350/358 refs  339s
reference cache built in 346s
reproduction of the baseline files, 20 rows, signed mean and max absolute difference:
  dino             mean -1.141e-05  max |d| 4.480e-05  n 20
  clip             mean +6.271e-06  max |d| 5.008e-05  n 20
  arcface_broad    mean +2.615e-04  max |d| 1.710e-03  n 19
  arcface_strict   mean +2.071e-04  max |d| 1.449e-03  n 6
embedding path reproduces the baseline files; ArcFace is rescored in section 4


## 4. Score the arms

One file per arm, `test__<arm>_identity.jsonl`, carrying both embedding and ArcFace columns.
Save-as-you-go and skip-if-exists: a dropped session resumes from the rows already written.
The baseline is not recomputed; `baseline_embed.jsonl` and `baseline_arcface.jsonl` stand.

In [4]:
# score files are written in thread-completion order, so the row index is the "i" field and
# never the line position
META = {r["i"]: r for r in (json.loads(l) for l in open(f"{SCORES}/test__baseline.jsonl", encoding="utf-8"))}
assert len(META) == len(rows), f"{len(META)} distinct i values, expected {len(rows)}"
assert all(META[i]["uid"] == UID[i] for i in META), "score-file uid does not match test.jsonl at the same i"

for lab, gdir, _ in ARMS:
    out = f"{SCORES}/test__{lab}_identity.jsonl"
    done = set()
    if os.path.exists(out):
        done = {json.loads(l)["i"] for l in open(out, encoding="utf-8")}
    todo = [i for i in range(len(rows)) if i not in done]
    print(f"\n{lab}: {len(done)} done, {len(todo)} to score", flush=True)
    t0 = time.time()
    with open(out, "a", encoding="utf-8") as f:
        for k, i in enumerate(todo):
            m = row_metrics(i, gen_path(gdir, i))
            f.write(json.dumps({"i": i, "uid": UID[i], "n_refs": META[i]["n_refs"],
                                "ref_kind": META[i]["ref_kind"], **m}) + "\n")
            if (k + 1) % 100 == 0:
                f.flush()
                print(f"  {k+1}/{len(todo)}  {time.time()-t0:.0f}s", flush=True)
    n = sum(1 for _ in open(out, encoding="utf-8"))
    assert n == len(rows), f"{lab}: {n} rows written, expected {len(rows)}"
    print(f"  {lab}: {n} rows in {time.time()-t0:.0f}s -> {os.path.basename(out)}")


baseline: 488 done, 0 to score
  baseline: 488 rows in 0s -> test__baseline_identity.jsonl

lora_r8_step1000: 488 done, 0 to score
  lora_r8_step1000: 488 rows in 0s -> test__lora_r8_step1000_identity.jsonl

lora_r8_step4000: 488 done, 0 to score
  lora_r8_step4000: 488 rows in 0s -> test__lora_r8_step4000_identity.jsonl

qlora_r8_step2000: 488 done, 0 to score
  qlora_r8_step2000: 488 rows in 0s -> test__qlora_r8_step2000_identity.jsonl

sp_iii_p139_step2000: 488 done, 0 to score
  sp_iii_p139_step2000: 488 rows in 0s -> test__sp_iii_p139_step2000_identity.jsonl


## 5. Result

Paired per-row deltas against the baseline on the same uid, for each measure, printed beside the
GPT-4.1 SC delta the arm already carries. Agreement in sign and ordering is the independent
confirmation; disagreement localises what SC measures that a similarity does not.

ArcFace is reported on its own coverage, which differs by arm only through the generated image,
since the reference side is fixed.

In [5]:
import math
from scipy import stats

# every arm including the baseline is read from this session's files, so no paired delta
# straddles the onnxruntime build change recorded in section 3
def load_id(lab):
    return {json.loads(l)["uid"]: json.loads(l)
            for l in open(f"{SCORES}/test__{lab}_identity.jsonl", encoding="utf-8")}

def load_gpt(sfile):
    return {json.loads(l)["uid"]: json.loads(l)
            for l in open(f"{SCORES}/{sfile}.jsonl", encoding="utf-8")}

BASE_ID, BASE_GPT = load_id("baseline"), load_gpt("test__baseline")

def paired(a, b, idx, key):
    d = np.array([a[k][key] - b[k][key] for k in idx], float)
    n = len(d); se = d.std(ddof=1) / math.sqrt(n)
    return n, d.mean(), se, stats.ttest_1samp(d, 0)[1]

print(f"{'arm':22s} {'measure':16s} {'n':>4s} {'delta':>9s} {'SE':>8s} {'p':>10s}")
for lab, _, sfile in ARMS:
    if lab == "baseline":
        continue
    A, G = load_id(lab), load_gpt(sfile)
    assert set(A) == set(BASE_ID), f"{lab}: uid sets differ from the baseline"
    for key in ("dino", "clip", "arcface_broad", "arcface_strict"):
        idx = sorted(k for k in A if A[k][key] is not None and BASE_ID[k][key] is not None)
        if len(idx) < 3:
            print(f"{lab:22s} {key:16s} {len(idx):>4d}  too few paired rows")
            continue
        n, m, se, p = paired(A, BASE_ID, idx, key)
        print(f"{lab:22s} {key:16s} {n:>4d} {m:>+9.4f} {se:>8.4f} {p:>10.3g}")
    idx = sorted(set(G) & set(BASE_GPT))
    n, m, se, p = paired(G, BASE_GPT, idx, "SC")
    print(f"{lab:22s} {'GPT-4.1 SC':16s} {n:>4d} {m:>+9.4f} {se:>8.4f} {p:>10.3g}")
    print()

print("by ref_kind, where the SC damage is concentrated:")
print(f"{'arm':22s} {'ref_kind':10s} {'n':>4s} {'dDINO':>9s} {'dCLIP':>9s} {'dArcF':>9s} {'dGPT SC':>9s}")
for lab, _, sfile in ARMS:
    if lab == "baseline":
        continue
    A, G = load_id(lab), load_gpt(sfile)
    for rk in ("no_up", "mixed", "all_up"):
        u = [k for k in A if A[k]["ref_kind"] == rk]
        cell = []
        for key in ("dino", "clip", "arcface_broad"):
            uu = [k for k in u if A[k][key] is not None and BASE_ID[k][key] is not None]
            cell.append(f"{np.mean([A[k][key]-BASE_ID[k][key] for k in uu]):+9.4f}" if len(uu) > 2 else f"{'n/a':>9s}")
        g = np.mean([G[k]["SC"] - BASE_GPT[k]["SC"] for k in u])
        print(f"{lab:22s} {rk:10s} {len(u):>4d} {cell[0]} {cell[1]} {cell[2]} {g:>+9.4f}")
    print()

# ArcFace needs a face in the reference, so its coverage varies by stratum while DINO and CLIP
# cover every row; this is where the instruments' reach differs from where the SC damage sits
print("ArcFace coverage by stratum, baseline:")
print(f"{'stratum':10s} {'n':>4s} {'broad':>11s} {'strict':>11s}")
for rk in ("no_up", "mixed", "all_up"):
    u = [k for k in BASE_ID if BASE_ID[k]["ref_kind"] == rk]
    nb_ = sum(1 for k in u if BASE_ID[k]["arcface_broad"] is not None)
    ns_ = sum(1 for k in u if BASE_ID[k]["arcface_strict"] is not None)
    print(f"{rk:10s} {len(u):>4d} {nb_:>4d} ({100*nb_/len(u):>3.0f}%) {ns_:>4d} ({100*ns_/len(u):>3.0f}%)")

# Level correlation asks whether a similarity can rank rows the way the judge ranks them.
# Delta correlation asks whether a row that the judge marks down is a row the similarity marks
# down. The arm-level agreement above establishes only that the two move together in the mean.
KEYS = ("dino", "clip", "arcface_broad", "arcface_strict")

print("row-level correlation of per-row deltas against the baseline:")
print(f"{'arm':22s} {'measure':16s} {'n':>5s} {'Pearson':>9s} {'Spearman':>9s} {'p':>10s}")
pool = {k: ([], []) for k in KEYS}
for lab, _, sfile in ARMS:
    if lab == "baseline":
        continue
    A, G = load_id(lab), load_gpt(sfile)
    for key in KEYS:
        idx = sorted(k for k in A if A[k][key] is not None and BASE_ID[k][key] is not None)
        x = np.array([A[k][key] - BASE_ID[k][key] for k in idx], float)
        y = np.array([G[k]["SC"] - BASE_GPT[k]["SC"] for k in idx], float)
        if len(idx) < 10 or x.std() == 0 or y.std() == 0:
            print(f"{lab:22s} {key:16s} {len(idx):>5d}  too few paired rows")
            continue
        pr, sp = stats.pearsonr(x, y), stats.spearmanr(x, y)
        # centred per arm, so the pooled figure below carries no between-arm effect-size variance
        pool[key][0].extend(x - x.mean())
        pool[key][1].extend(y - y.mean())
        print(f"{lab:22s} {key:16s} {len(idx):>5d} {pr[0]:>+9.3f} {sp[0]:>+9.3f} {pr[1]:>10.3g}")
    print()

print("pooled within-arm, each arm centred before pooling:")
for key in KEYS:
    x, y = np.array(pool[key][0]), np.array(pool[key][1])
    if len(x) < 10:
        continue
    pr, sp = stats.pearsonr(x, y), stats.spearmanr(x, y)
    print(f"  {key:16s} n {len(x):>5d}  Pearson {pr[0]:>+.3f}  Spearman {sp[0]:>+.3f}  p {pr[1]:.3g}")

print("\nfor contrast, the same measures against the GPT SC level on the baseline alone:")
for key in KEYS:
    idx = sorted(k for k in BASE_ID if BASE_ID[k][key] is not None)
    x = np.array([BASE_ID[k][key] for k in idx], float)
    y = np.array([BASE_GPT[k]["SC"] for k in idx], float)
    pr, sp = stats.pearsonr(x, y), stats.spearmanr(x, y)
    print(f"  {key:16s} n {len(x):>5d}  Pearson {pr[0]:>+.3f}  Spearman {sp[0]:>+.3f}  p {pr[1]:.3g}")

arm                    measure             n     delta       SE          p
lora_r8_step1000       dino              488   -0.0063   0.0013   1.07e-06
lora_r8_step1000       clip              488   -0.0006   0.0009      0.485
lora_r8_step1000       arcface_broad     376   -0.0294   0.0033   1.67e-17
lora_r8_step1000       arcface_strict    106   -0.0389   0.0067   7.01e-08
lora_r8_step1000       GPT-4.1 SC        488   -0.2418   0.0690     0.0005

lora_r8_step4000       dino              488   -0.0220   0.0027   3.09e-15
lora_r8_step4000       clip              488   -0.0083   0.0017   8.58e-07
lora_r8_step4000       arcface_broad     375   -0.0823   0.0060   5.04e-35
lora_r8_step4000       arcface_strict    104   -0.1020   0.0128   2.04e-12
lora_r8_step4000       GPT-4.1 SC        488   -0.6557   0.1003   1.57e-10

qlora_r8_step2000      dino              488   -0.0066   0.0017   8.97e-05
qlora_r8_step2000      clip              488   -0.0002   0.0011      0.855
qlora_r8_step2000      